# EDA: Planning London Datahub, Brent and Lewisham

Goal: understand the data before building endpoints on it.

**Golden rule:** only Brent and Lewisham. Every query in this notebook carries the scope filter.

## Base URL

The guest (read-only) endpoint of the Planning London Datahub. It's an Elasticsearch 7.9 cluster, so each index is a path under this URL, e.g. `applications/_search`.

In [ ]:
BASE_URL = "https://planningdata.london.gov.uk/api-guest"

## Access header

Every request needs `X-API-AllowRequest`. The value is read from the project's `.env` (`DATAHUB_ALLOW_REQUEST`), so it is never pasted into the notebook.

In [ ]:
from pathlib import Path

from dotenv import dotenv_values

# The notebook runs from findings/, so walk up to the project root (where pyproject.toml is).
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

HEADERS = {
    "X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"],
    "Content-Type": "application/json",
}

## Brent filter

Scope is by **location**: `borough` is Brent, or `borough` is missing and Brent decided it (`lpa_name`). This also includes the OPDC-decided sites in Brent. Every query below uses this.

In [ ]:
def brent_filter() -> dict:
    """Elasticsearch clause matching applications located in Brent."""
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {
                    "bool": {
                        "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                        "must_not": [{"exists": {"field": "borough.raw"}}],
                    }
                },
            ],
            "minimum_should_match": 1,
        }
    }

## 10 Brent applications received in 2026

"2026" here means `valid_date` (when the application was received and validated). Dates are `DD/MM/YYYY`, so the range says so explicitly. Newest first.

In [ ]:
import httpx

query = {
    "size": 10,
    "track_total_hits": True,
    "query": {
        "bool": {
            "filter": [
                brent_filter(),
                {"range": {"valid_date": {"gte": "01/01/2026", "lte": "31/12/2026", "format": "dd/MM/yyyy"}}},
            ]
        }
    },
    "sort": [{"valid_date": "desc"}],
}

response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=query, timeout=30)
response.raise_for_status()
result = response.json()

total = result["hits"]["total"]["value"]
docs = [hit["_source"] for hit in result["hits"]["hits"]]
print(f"Brent applications received in 2026: {total}; showing {len(docs)}")

### A compact view

A few key fields per application. Each document has many more (see the next cell).

In [ ]:
for d in docs:
    print(
        f"{d['id']:<22} valid {d.get('valid_date')}  decided_by {d.get('lpa_name'):<6} "
        f"ward {str(d.get('ward')):<22} {str(d.get('application_type')):<15} "
        f"status {str(d.get('status')):<32}"
    )
    print(f"    {str(d.get('development_type')):<35} {str(d.get('description'))[:110]}")

### One full document

The first application as the API returns it, to see every field that is filled in.

In [ ]:
import json

print(json.dumps(docs[0], indent=2)[:6000])

## The same 10 as CSV (SQL endpoint)

Elasticsearch also has an SQL endpoint, `_sql`, and `?format=csv` makes it answer in CSV. Guests are allowed to use it (checked 2026-11-06).

- The Brent rule is the same as `brent_filter()`, written in SQL.
- Dates in the `WHERE` clause must be `DD/MM/YYYY`, the server's stored format (`'2026-01-01'` gives a parse error). In the CSV they come back as ISO timestamps.
- `SELECT *` flattens each document into ~211 columns (e.g. `parking_details.net_car`).
- **Lists inside a document are left out**: individual homes, floorspace rows, buildings, phasing. Those come from their own indexes (e.g. `residential_units`).

In [ ]:
# Same rule as brent_filter(): location if known, otherwise who decided it.
BRENT_SQL = "(borough = 'Brent' OR (lpa_name = 'Brent' AND borough IS NULL))"

sql = f"""
SELECT *
FROM applications
WHERE {BRENT_SQL}
  AND valid_date >= '01/01/2026' AND valid_date <= '31/12/2026'
ORDER BY valid_date DESC
LIMIT 10
"""

response = httpx.post(
    f"{BASE_URL}/_sql", params={"format": "csv"}, headers=HEADERS, json={"query": sql}, timeout=30
)
response.raise_for_status()

DATA_DIR = PROJECT_ROOT / "findings" / "data"
DATA_DIR.mkdir(exist_ok=True)
csv_path = DATA_DIR / "brent_2026_latest10.csv"
csv_path.write_text(response.text, encoding="utf-8")
print(f"Saved {csv_path.relative_to(PROJECT_ROOT)} ({len(response.text):,} characters)")

### Check the CSV

Row and column count, plus a few columns to compare with the JSON view above.

In [ ]:
import csv

with csv_path.open(encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"{len(rows)} rows x {len(rows[0])} columns")
for r in rows:
    print(f"{r['id']:<16} {r['valid_date'][:10]}  {r['borough']:<6} {r['ward']:<26} {r['status']}")

# Data quality EDA: all Brent applications

Goal: find the data-quality problems to handle **before** loading Brent `applications` into Postgres.

## 1. Download every Brent application (raw JSON)

- **Why not year slices:** 3,928 Brent records have no `valid_date`, so slicing by year would silently drop them.
- **Why not `from`/`size`:** paging stops at 10,000 and Brent has ~37k.
- **`search_after`:** sort by `id` (unique, exact-match) and ask for "the next 1,000 after the last id I got". It has no depth limit.
- **One JSON document per line (JSONL),** unchanged, so nothing is lost (nested lists included). This is also what goes into Postgres.
- **Already downloaded?** Then it's reused. Set `REFRESH = True` to fetch again (the data changes daily).

In [ ]:
import json

REFRESH = False
RAW_PATH = DATA_DIR / "brent_applications.jsonl"


def count_brent() -> int:
    """Number of Brent applications according to the API."""
    body = {"query": {"bool": {"filter": [brent_filter()]}}}
    r = httpx.post(f"{BASE_URL}/applications/_count", headers=HEADERS, json=body, timeout=60)
    r.raise_for_status()
    return r.json()["count"]


def download_brent(path: Path, page_size: int = 1000) -> int:
    """Write every Brent application to `path` as JSON lines; return how many were written."""
    written, after = 0, None
    with httpx.Client(headers=HEADERS, timeout=120) as client, path.open("w", encoding="utf-8") as out:
        while True:
            body = {
                "size": page_size,
                "query": {"bool": {"filter": [brent_filter()]}},
                "sort": [{"id": "asc"}],
            }
            if after is not None:
                body["search_after"] = after
            r = client.post(f"{BASE_URL}/applications/_search", json=body)
            r.raise_for_status()
            hits = r.json()["hits"]["hits"]
            if not hits:
                return written
            for hit in hits:
                out.write(json.dumps(hit["_source"], ensure_ascii=False) + "\n")
            written += len(hits)
            after = hits[-1]["sort"]


expected = count_brent()
if REFRESH or not RAW_PATH.exists():
    n = download_brent(RAW_PATH)
    print(f"Downloaded {n:,} applications")
print(f"API count: {expected:,}; file: {RAW_PATH.relative_to(PROJECT_ROOT)} "
      f"({RAW_PATH.stat().st_size / 1e6:.1f} MB)")

## 2. Load into pandas

Top-level fields become columns. Nested objects (`application_details`, `parking_details`, `centroid`, polygons) stay as Python objects in their column for now.

In [ ]:
import pandas as pd

pd.set_option("display.max_rows", 200, "display.max_colwidth", 80, "display.width", 200)

with RAW_PATH.open(encoding="utf-8") as f:
    docs = [json.loads(line) for line in f]

df = pd.DataFrame(docs)
print(f"{len(df):,} rows x {df.shape[1]} columns (API count was {expected:,})")

## 3. Identity

Checks: is `id` unique? Is it really `<lpa_name>-<lpa_app_no with / replaced by _>`?
If not, which other characters get replaced?

In [ ]:
import re

print("duplicate ids:", df["id"].duplicated().sum())
print("missing ids:  ", df["id"].isna().sum())

expected_id = df["lpa_name"] + "-" + df["lpa_app_no"].fillna("").str.replace("/", "_")
mismatch = df[df["id"] != expected_id]
print(f"id != lpa_name-lpa_app_no(/→_): {len(mismatch):,}")
print(mismatch[["id", "lpa_name", "lpa_app_no"]].head(15).to_string())

# Characters that appear in references at all
chars = sorted({c for ref in df["lpa_app_no"].dropna() for c in ref if not c.isalnum()})
print("non-alphanumeric characters in lpa_app_no:", chars)

## 4. Scope sanity check (golden rule)

Every row must be located in Brent, or have no location and be decided by Brent.

In [ ]:
combo = df.groupby([df["lpa_name"].fillna("(none)"), df["borough"].fillna("(none)")]).size()
print(combo.to_string())

in_scope = (df["borough"] == "Brent") | (df["borough"].isna() & (df["lpa_name"] == "Brent"))
print("\nrows failing the Brent rule:", (~in_scope).sum())

## 5. Completeness of top-level fields

"Missing" = `null`, empty string, whitespace only or empty list. Sorted from emptiest. This shows which columns are worth loading and which will almost always be NULL.

In [ ]:
def is_missing(v) -> bool:
    if v is None:
        return True
    if isinstance(v, float) and pd.isna(v):
        return True
    if isinstance(v, str) and not v.strip():
        return True
    if isinstance(v, (list, dict)) and not v:
        return True
    return False


missing = df.apply(lambda col: col.map(is_missing).mean() * 100).sort_values(ascending=False)
print(missing.round(1).to_string())

## 6. Dates

For every date-like column, which **formats** appear? Then which values can't be parsed, which are in the future, and which are implausibly old.

In [ ]:
DATE_COLS = [c for c in df.columns if c.endswith("_date") or c in ("last_updated", "last_synced", "pp_retry_expiry")]

PATTERNS = {
    "DD/MM/YYYY": re.compile(r"^\d{2}/\d{2}/\d{4}$"),
    "ISO timestamp": re.compile(r"^\d{4}-\d{2}-\d{2}T"),
    "YYYY-MM-DD": re.compile(r"^\d{4}-\d{2}-\d{2}$"),
    "MM/YYYY": re.compile(r"^\d{2}/\d{4}$"),
}


def date_format(v) -> str:
    if is_missing(v):
        return "missing"
    for name, rx in PATTERNS.items():
        if rx.match(str(v)):
            return name
    return f"OTHER: {v!r}"


fmt = {c: df[c].map(date_format).value_counts() for c in DATE_COLS}
print(pd.DataFrame(fmt).fillna(0).astype(int).T.to_string())

In [ ]:
def parse_dmy(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")


NOW = pd.Timestamp.now()
rows = []
for c in DATE_COLS:
    present = df[c].map(lambda v: not is_missing(v))
    if df[c].map(date_format).eq("ISO timestamp").any():
        parsed = pd.to_datetime(df[c], errors="coerce", utc=True).dt.tz_localize(None)
    else:
        parsed = parse_dmy(df[c])
    rows.append({
        "column": c,
        "present": int(present.sum()),
        "unparseable": int((present & parsed.isna()).sum()),
        "min": parsed.min(),
        "max": parsed.max(),
        "future": int((parsed > NOW).sum()),
        "before_1990": int((parsed < "1990-01-01").sum()),
    })
print(pd.DataFrame(rows).to_string(index=False))

### Date order

A case should move forward in time: valid → decision → commencement → completion. How often does it go backwards?

In [ ]:
d = {c: parse_dmy(df[c]) for c in ["valid_date", "decision_date", "decision_target_date",
                                   "actual_commencement_date", "actual_completion_date", "lapsed_date"]}

checks = {
    "decision before valid": d["decision_date"] < d["valid_date"],
    "target before valid": d["decision_target_date"] < d["valid_date"],
    "commencement before decision": d["actual_commencement_date"] < d["decision_date"],
    "completion before commencement": d["actual_completion_date"] < d["actual_commencement_date"],
    "lapsed before decision": d["lapsed_date"] < d["decision_date"],
    "decision date but no decision": d["decision_date"].notna() & df["decision"].map(is_missing),
    "decision but no decision date": d["decision_date"].isna() & ~df["decision"].map(is_missing),
}
print(pd.Series({k: int(v.sum()) for k, v in checks.items()}).to_string())

days = (d["decision_date"] - d["valid_date"]).dt.days
print("\ndays from valid to decision:")
print(days.describe().round(1).to_string())

## 7. Categorical values

Distinct values with counts, using `repr` so hidden whitespace (`'\tApproved'`) is visible. Then the values that collide after `strip()` + lower-case, i.e. the same thing spelled differently.

In [ ]:
CAT_COLS = ["status", "decision", "application_type", "application_type_full", "development_type",
            "decision_process", "decision_agency", "appeal_status", "appeal_decision", "bo_system",
            "last_updated_by"]

for c in CAT_COLS:
    vc = df[c].map(lambda v: "<missing>" if is_missing(v) else repr(v)).value_counts()
    print(f"\n### {c}: {len(vc)} distinct")
    print(vc.head(40).to_string())

In [ ]:
def collisions(col: str) -> dict:
    """Groups of different raw values that are equal after strip + lower-case."""
    vals = df[col].dropna().astype(str)
    groups = vals.groupby(vals.str.strip().str.casefold()).agg(lambda s: sorted(set(s)))
    return {k: v for k, v in groups.items() if len(v) > 1}


for c in CAT_COLS + ["ward"]:
    found = collisions(c)
    if found:
        print(f"{c}: {found}")

### Status vs decision

Do `status` and `decision` agree? (e.g. status Approved but decision Refused)

In [ ]:
ct = pd.crosstab(df["status"].fillna("<missing>"), df["decision"].fillna("<missing>"))
print(ct.to_string())

## 8. Wards

Brent has 22 wards (since May 2022). Older or misspelled names show up here.

In [ ]:
print(df["ward"].map(lambda v: "<missing>" if is_missing(v) else v).value_counts().to_string())

## 9. Text fields

Leading/trailing whitespace, tabs, newlines and non-breaking spaces in every string column. These break exact matching and grouping.

In [ ]:
str_cols = [c for c in df.columns if df[c].map(lambda v: isinstance(v, str)).any()]
issues = []
for c in str_cols:
    s = df[c][df[c].map(lambda v: isinstance(v, str))]
    issues.append({
        "column": c,
        "strings": len(s),
        "leading/trailing ws": int((s != s.str.strip()).sum()),
        "tab/newline": int(s.str.contains(r"[\t\r\n]").sum()),
        "nbsp": int(s.str.contains("\u00a0").sum()),
        "empty": int((s.str.strip() == "").sum()),
    })
iss = pd.DataFrame(issues).set_index("column")
print(iss[iss.drop(columns="strings").sum(axis=1) > 0].to_string())

## 10. Location

Is every centroid present and inside Brent? Rough Brent bounding box: lat 51.52–51.60, lon −0.34 to −0.19. Easting/northing should agree with lat/lon.

In [ ]:
lat = df["centroid"].map(lambda c: c.get("lat") if isinstance(c, dict) else None).astype(float)
lon = df["centroid"].map(lambda c: c.get("lon") if isinstance(c, dict) else None).astype(float)

# 49.766807, -7.55716 is the origin of the British National Grid (easting 0, northing 0),
# i.e. "no location" converted to lat/lon, not a real place.
placeholder = lat.round(4).eq(49.7668) & lon.round(4).eq(-7.5572)
inside = lat.between(51.52, 51.60) & lon.between(-0.34, -0.19)

print("centroid missing:              ", int(lat.isna().sum()))
print("centroid = grid-origin placeholder:", int(placeholder.sum()))
print("other centroids outside Brent box: ", int((lat.notna() & ~inside & ~placeholder).sum()))
print(df.loc[lat.notna() & ~inside & ~placeholder, ["id", "lpa_name", "ward", "site_name"]]
      .assign(lat=lat, lon=lon).head(15).to_string())

### Easting / northing types

These should be numbers, but some are stored as strings. Which types and odd values occur?

In [ ]:
for c in ["centroid_easting", "centroid_northing"]:
    types = df[c].map(lambda v: type(v).__name__).value_counts()
    num = pd.to_numeric(df[c], errors="coerce")
    bad = df[c][df[c].notna() & num.isna()]
    print(f"{c}: types {types.to_dict()}")
    print(f"   numeric range {num.min()} - {num.max()}; zero: {int((num == 0).sum())}; "
          f"not numeric: {len(bad)} e.g. {bad.unique()[:5].tolist()}")

### Odd wards

Ward values that are not a current or former Brent ward name.

In [ ]:
odd = df[df["ward"].isin(["East Acton", "COLLP", "Wembley"])]
print(odd[["id", "lpa_name", "borough", "ward", "site_name"]].to_string())

## 11. Links

Do `url_planning_app` values point at the expected portals?

In [ ]:
host = df["url_planning_app"].dropna().str.extract(r"^(https?://[^/]+)")[0]
print(host.value_counts(dropna=False).to_string())
print("missing:", int(df["url_planning_app"].map(is_missing).sum()))

## 12. How often the rich detail exists

`application_details` (housing, floorspace, infrastructure…) by application type.

In [ ]:
has_details = df["application_details"].map(lambda v: isinstance(v, dict) and bool(v))
print(pd.crosstab(df["application_type"].fillna("<missing>"), has_details, margins=True).to_string())